# Thesis 2027: Pursuit-Evasion Benchmark

This project provides a small, reproducible benchmark for comparing cooperative pursuers in grid-based pursuit-evasion games. It includes difficulty-controlled level generation, pure pursuit/reach-avoid/maze modes, classical planner entry points, MARL training drivers, metrics, ASCII rendering, and notebook workflows.

## Quick start

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python main.py demo
python main.py benchmark
python main.py train --algorithm QMIX --episodes 100
python main.py notebook
```

Open the notebooks in `notebooks/` from top to bottom. The notebooks are deliberately thin: reusable experiment logic lives in `env/`, `planners/`, `marl.py`, `scenarios.py`, and `utils.py`.

## Project map

- `env/`: scenario generation and the shared environment API.
- `planners/`: CBS-style search, M*, Voronoi, and a time-expanded extension hook.
- `marl.py`: compatible training-driver interface for VDN, QMIX, QTRAN, and QPLEX baselines.
- `main.py`: simple command-line entry point for demos, benchmarks, training, and notebooks.
- `configs/`: example difficulty specifications.
- `notebooks/`: demonstrations, training, benchmarking, analysis, and UI preview.
- `ui.py`: dependency-free ASCII renderer suitable for early validation.

Running `python main.py` or `python main.py demo` opens the Tkinter simulation window. The pursuers and evader then move automatically; use Pause, Reset episode, or Close in the window.

The application flow is Main Menu → Game Mode → Algorithm → Difficulty → Simulation. Walls are directional barriers drawn as blue lines and still allow an agent to stand on their tile. Barrels are brown impassable cells. Maze scenarios generate walls only and never generate barrels.

The popup includes three selectable levels. Every agent moves at one fixed cell per turn:

| Level | Grid | Pursuers | Objects |
| --- | --- | --- | --- |
| 1 | 5 x 5 | 1 | None |
| 2 | 10 x 10 | 2 | Walls and barrels |
| 3 | 20 x 20 | 3 | Many walls and barrels |

## Research workflow

## Primary benchmark specification

The primary comparison is cooperative Reach-Avoid on a generated maze. A state is
the grid dimensions, directional walls, barrels, pursuer positions, evader
positions, goals, and timestep. Pursuers choose one movement action each turn
and share a reward; the evader chooses independently. The pursuers succeed when
they capture the evader. The evader succeeds when it reaches a goal, while a
horizon is reported as truncation rather than termination.

Every episode is generated from a held-out seed. Generation rejects boards that
disconnect any start or goal from the board's traversable component or place an
evader within one move of a pursuer. Evaluation uses the shared rollout API and
reports capture rate with a 95% confidence interval, time to terminal event,
path efficiency, and compute time. Per-agent observations include the complete
legal-action mask; the global state is a fixed-length grid encoding for value
mixers.

1. Establish environment and reward sanity in `01_environment_demo.ipynb`.
2. Add planner implementations and evaluate them on fixed scenarios.
3. Replace the lightweight MARL driver with the selected PyTorch/PyMARL implementation when the observation/action contract is stable.
4. Keep held-out scenarios for `04_benchmark_evaluation.ipynb` and report capture rate, time-to-capture, path efficiency, planning/training compute, and scaling.

The current planners and MARL drivers are intentionally small baselines, not claims of complete research-grade CBS, M*, Voronoi defense, or deep value-decomposition implementations. That separation keeps the project runnable while leaving a clean place to add the algorithms and citations required by the thesis.


These are some of the suggestions to implement (Work on them in order from 1 - 5):

1.   Fix the environment.
Seed the collision randomness, guarantee solvable and fair scenarios, fix the evader reward, separate termination from truncation, and add tests. 

Why: 
every result depends on the environment. Right now identical seeds can give different episodes, 11–19% of Maze scenarios are unsolvable, and the evader earns more by being caught early than by surviving (-1.09 vs -2.0). Results gathered on a flawed or changing environment would have to be thrown away.

2.   Write the benchmark spec and design observations.
Pick one primary benchmark (cooperative Reach-Avoid on a maze), define it formally, and build per-agent observations, a global state and action masks. 

**Why:** 
The brief explicitly asks you to model the right state information, and a chosen, formally defined problem is what makes the comparison meaningful. The learners also cannot be trained without observations, masks and a global state for the mixers.

3.   **Build an evaluation harness.** 
One rollout function for any policy, with capture rate, time-to-capture, path efficiency and compute, using frozen held-out seeds and confidence intervals. 

**Why:** 
The project is a comparison. Today planners are never run through env.step, success is always True, and each level uses a single seed, so nothing can be measured fairly. One shared harness makes classical and MARL results directly comparable.

4.  Implement real classical planners.
Voronoi pursuit first, then CBS (plus ICBS or ECBS) and M*, wrapped in a receding-horizon loop. 

Why: 
The brief names these algorithms, and the current ones are stubs (M* is CBS renamed, Voronoi ignores walls). A receding-horizon wrapper is needed because CBS and M* assume fixed goals while the evader moves. Starting with Voronoi gives a strong, cheap baseline quickly.

5.   Implement real MARL
VDN, QMIX, QTRAN and QPLEX in PyTorch (plus IQL as a baseline), with a shared training loop, logging,
checkpoints and a curriculum that switches levels. 

Why:
The current marl.py is a synthetic curve that gives identical results for all four algorithms, so it says nothing about them. A shared loop ensures the algorithms differ only in how they mix Q-values, and IQL shows how much value decomposition actually adds.